# [DistilBERT, a distilled verison of BERT: smaller, faster, cheaper and lighter](https://arxiv.org/pdf/1910.01108)

## Introduction

**Gap:** Pre-trained language models like BERT achieve strong performance but are large, slow, and memory-intensive, making them difficult to deploy in real-world or resource-constrained settings.

Prior knowledge distillation approaches mainly focus on task-specific compression **after fine-tuning**, which limits reusability and does not produce a general-purpose efficient pretrained model.

**Improvement**: DistilBERT proposes a method to pretrain a smaller transformer model using knowledge distillation from BERT, producing a foundational model that retains most of BERT’s performance while being significantly more efficient for both inference and downstream fine-tuning.

## Approach

DistilBERT trains a smaller “student” transformer to **mimic** a frozen pretrained BERT “teacher” using **multiple complementary losses**:

* **Language Modeling Loss (MLM):**
The student is trained on masked language modeling, similar to BERT, ensuring it still learns meaningful token-level representations from raw text
* **Distillation Loss (soft-target matching):**
The student matches the teacher’s output distribution (softmax probabilities over vocabulary)
* **Cosine Embedding / Hidden-State Alignment Loss**:
The student is encouraged to align its internal hidden representations with those of the teacher by maximizing cosine similarity between corresponding token-level hidden states.
This acts as an intermediate-level constraint, pushing the student not just to match outputs but also to learn similar feature spaces.


## Result

DistilBERT retains approximately 97% of BERT’s NLU performance while being ~40% smaller and ~60% faster at inference time.

## Application

In [8]:
import pandas as pd
df = pd.read_csv("data/toxicity_en.csv")
df

,text,is_toxic
0,"Elon Musk is a piece of shit, greedy capitalis...",Toxic
1,The senile credit card shrill from Delaware ne...,Toxic
2,He does that a lot -- makes everyone look good...,Toxic
3,F*ck Lizzo,Toxic
4,Epstein and trump were best buds!!! Pedophiles...,Toxic
...,...,...
995,My maternal abuelita taught me how to make pla...,Not Toxic
996,Funnily enough I was looking online last week ...,Not Toxic
997,I can't bear how nice this is.\n \n I guess it...,Not Toxic
998,Going to buy a share of Tesla just to ensure i...,Not Toxic


In [9]:
df.is_toxic.value_counts()

is_toxic
Toxic        501
Not Toxic    499
Name: count, dtype: int64

In [10]:
df['is_toxic'] = (df['is_toxic'] == 'Toxic').astype(int)
df

,text,is_toxic
0,"Elon Musk is a piece of shit, greedy capitalis...",1
1,The senile credit card shrill from Delaware ne...,1
2,He does that a lot -- makes everyone look good...,1
3,F*ck Lizzo,1
4,Epstein and trump were best buds!!! Pedophiles...,1
...,...,...
995,My maternal abuelita taught me how to make pla...,0
996,Funnily enough I was looking online last week ...,0
997,I can't bear how nice this is.\n \n I guess it...,0
998,Going to buy a share of Tesla just to ensure i...,0


In [11]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:
from transformers import AutoTokenizer, AutoModel

# Load tokenizer and model (recommended generic classes)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased").to(device)

text = "Replace me by any text you'd like."

# Tokenize
inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(device) for k, v in inputs.items()}

# Forward pass (no grad for inference)
with torch.no_grad():
    outputs = model(**inputs)

# Outputs
last_hidden_state = outputs.last_hidden_state
pooler_output = outputs.pooler_output  # if available

print(last_hidden_state.shape)
print(pooler_output.shape)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

C:\Users\angel\anaconda3\envs\efficiency\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\angel\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 12, 768])
torch.Size([1, 768])


In [13]:
import torch.nn as nn 

class ClassifierHead(nn.Module):
    def __init__(self, hidden_size=768, output_size=1):
        super().__init__()
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        return self.linear(x)

classifier = ClassifierHead().to(device)
classifier(last_hidden_state.mean(axis=1))

tensor([[0.1755]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [14]:
from torch.utils.data import Dataset, DataLoader

class ToxicDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.texts = df["text"].values
        self.labels = df["is_toxic"].values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": label
        }

        return item

In [17]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["is_toxic"]  # important for class balance
)

train_dataset = ToxicDataset(train_df, tokenizer, max_length=128)
test_dataset = ToxicDataset(test_df, tokenizer, max_length=128)

from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

batch = next(iter(train_dataloader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)
print()

batch = next(iter(test_dataloader))

print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])

torch.Size([16, 128])
torch.Size([16, 128])
torch.Size([16])


In [18]:
from tqdm import tqdm

def test_bert(model, classifier, criterion):
    model.eval()
    classifier.eval()

    total_loss = 0
    correct = 0
    total = 0

    loop = tqdm(test_dataloader, desc="Testing", leave=True)

    with torch.no_grad():
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            # forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            cls_embedding = outputs.last_hidden_state[:, 0, :]
            logits = classifier(cls_embedding).squeeze(-1)

            loss = criterion(logits, labels)
            total_loss += loss.item()

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(test_dataloader)
    accuracy = correct / total
    return avg_loss, accuracy
    
def train_bert(model, classifier, optimizer, criterion, epochs=30):
    model.train()
    classifier.train()

    for epoch in range(epochs):
        total_loss = 0

        loop = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{epochs}", leave=True)

        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].float().to(device)

            # forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            # CLS token representation
            cls_embedding = outputs.last_hidden_state[:, 0, :]

            logits = classifier(cls_embedding).squeeze(-1)

            loss = criterion(logits, labels)

            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_value = loss.item()
            total_loss += loss_value

            loop.set_postfix(loss=loss_value)

        avg_loss = total_loss / len(train_dataloader)
        test_loss, test_acc = test_bert(model, classifier, criterion)
        print(f"Epoch {epoch+1}/{epochs} Train loss: {avg_loss:.4f} | Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}\n")

    return model, classifier

In [19]:
from torch import nn
from torch import optim

LR = 3e-5
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(list(model.parameters()) + list(classifier.parameters()), lr=LR)

In [20]:
train_bert(model, classifier, optimizer, criterion)

Epoch 1/30:  26%|███████████████▎                                           | 14/54 [00:07<00:21,  1.90it/s, loss=0.38]


KeyboardInterrupt: 

In [21]:
from transformers import DistilBertConfig, DistilBertModel

config = DistilBertConfig()
small_model = DistilBertModel(config) 

In [22]:
model

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
small_model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L